In [0]:
catalog = dbutils.widgets.get("catalog");
spark.sql(f"USE CATALOG {catalog}");
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

Phase d'agregation

In [0]:
from pyspark.sql.functions import *

In [0]:
# Retard moyen par mois

gold_delay_month = (
    spark.table("silver.silver_city_flights")
    .groupBy("numberMonth","month_name")
    .agg(
        round(avg("delay").alias("avg_delay"),2).alias("avg_delay"),
        count("*").alias("nb_flights")
    )
)

gold_delay_month.show()


In [0]:
gold_delay_month.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_delay_month")

In [0]:
# Retard moyen par ville

gold_delay_town =(
spark.table("silver.silver_city_flights")
.groupBy("origin_town")
.agg(
    round(avg("delay").alias("avg_delay"),2).alias("avg_delay")
    )

)

gold_delay_town.show(5)


In [0]:
gold_delay_town.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_delay_town")

In [0]:
# Frequence des trajets

gold_trips_freq = (
spark.table("silver.silver_city_flights")
.groupBy("origin_town","destination_town")
.agg(
    count("*").alias("nb_trips")
    )
.orderBy("nb_trips", ascending=False)
)

gold_trips_freq.show(5)


In [0]:
gold_trips_freq.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_trips_freq")

In [0]:
# Les villes de destination les plus fréquente
gold_dest_freq = (
spark.table("silver.silver_city_flights")
.groupBy("destination_town")
.agg(
    count("*").alias("nb_trips")
    )
.orderBy("nb_trips", ascending=False)
)

gold_dest_freq.show(5)

In [0]:
gold_dest_freq.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_dest_freq")

In [0]:
# Les villes de depart les plus fréquentes

gold_origin_freq = (
spark.table("silver.silver_city_flights")
.groupBy("origin_town")
.agg(
    count("*").alias("nb_trips")
    )
.orderBy("nb_trips", ascending=False)
)

gold_origin_freq.show(5)

In [0]:
gold_origin_freq.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_origin_freq")

In [0]:
gold_rejet = spark.table("flights.silver.silver_depart_rejet")
gold_rejet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_rejet")